# Decision Trees & Random Forest

**When:** Week 13 · Sessions B–C  
**Goal:** Learn nonlinear models that are strong baselines for tabular data.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.datasets import load_breast_cancer


## Intuition
- A **decision tree** asks yes/no questions about features (like a flowchart).
- Easy to visualize, can overfit if grown too deep.
- A **Random Forest** = many trees on random subsets → usually more accurate and stable.


In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


## 1. Decision Tree

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=42)
tree.fit(X_train, y_train)
print(classification_report(y_test, tree.predict(X_test), target_names=data.target_names))

plt.figure(figsize=(14, 6))
plot_tree(tree, feature_names=feature_names, class_names=list(data.target_names), filled=True, fontsize=7)
plt.title("Decision Tree (max_depth=3)")
plt.show()


## 2. Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
print(classification_report(y_test, rf.predict(X_test), target_names=data.target_names))

imp = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False).head(10)
imp.plot(kind="barh", figsize=(7, 4), title="Top feature importances (Random Forest)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 3. Compare with cross-validation

In [ ]:
for name, model in [("Tree depth=3", tree), ("Random Forest", rf)]:
    scores = cross_val_score(model, X, y, cv=5, scoring="f1")
    print(f"{name:16s} F1={scores.mean():.3f} +/- {scores.std():.3f}")


## 4. Class dataset practice (heart / diabetes)

In [ ]:
heart_path = Path("../data_cleaningML/data_cleaning/heart.csv")
diabetes_path = Path("../data_cleaningML/data_cleaning/diabetes.csv")

path = heart_path if heart_path.exists() else diabetes_path
if path.exists():
    df = pd.read_csv(path)
    print("Loaded:", path.name)
    print(df.head())
    # Common pattern: last column is often the target — confirm before training
    target = df.columns[-1]
    Xc = df.drop(columns=[target])
    yc = df[target]
    # Encode non-numeric if needed
    Xc = pd.get_dummies(Xc, drop_first=True)
    Xtr, Xte, ytr, yte = train_test_split(Xc, yc, test_size=0.2, random_state=42, stratify=yc if yc.nunique() < 20 else None)
    model = RandomForestClassifier(n_estimators=200, random_state=42)
    model.fit(Xtr, ytr)
    print(classification_report(yte, model.predict(Xte)))
    ConfusionMatrixDisplay.from_estimator(model, Xte, yte)
    plt.show()
else:
    print("Class CSVs not found — breast cancer demo above is enough for class.")


## Takeaways
- Start with Logistic Regression (interpretable baseline).
- Add Tree / Random Forest when relationships are nonlinear.
- Always compare with the **same metrics** and preferably **cross-validation**.
- Report feature importances carefully — they are clues, not proof of causation.
